In [161]:
import pandas as pd
import re

In [162]:
emails_df = pd.read_csv('../data/01_extracted_emails.csv')
print(emails_df.head(5))

                                    Message-ID  \
0  18782981.1075855378110.JavaMail.evans@thyme   
1  15464986.1075855378456.JavaMail.evans@thyme   
2  24216240.1075855687451.JavaMail.evans@thyme   
3  13505866.1075863688222.JavaMail.evans@thyme   
4  30922949.1075863688243.JavaMail.evans@thyme   

                                    Date                     From  \
0  Mon, 14 May 2001 16:39:00 -0700 (PDT)  phillip.allen@enron.com   
1   Fri, 4 May 2001 13:51:00 -0700 (PDT)  phillip.allen@enron.com   
2  Wed, 18 Oct 2000 03:00:00 -0700 (PDT)  phillip.allen@enron.com   
3  Mon, 23 Oct 2000 06:13:00 -0700 (PDT)  phillip.allen@enron.com   
4  Thu, 31 Aug 2000 05:07:00 -0700 (PDT)  phillip.allen@enron.com   

                        To    Subject   Cc  Mime-Version  \
0     tim.belden@enron.com        NaN  NaN           1.0   
1  john.lavorato@enron.com        Re:  NaN           1.0   
2   leah.arsdall@enron.com   Re: test  NaN           1.0   
3    randall.gay@enron.com        NaN  NaN  

In [163]:
emails_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 517401 entries, 0 to 517400
Data columns (total 18 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   Message-ID                 495554 non-null  object 
 1   Date                       495554 non-null  object 
 2   From                       495554 non-null  object 
 3   To                         495554 non-null  object 
 4   Subject                    478886 non-null  object 
 5   Cc                         124262 non-null  object 
 6   Mime-Version               495554 non-null  float64
 7   Content-Type               495554 non-null  object 
 8   Content-Transfer-Encoding  495554 non-null  object 
 9   Bcc                        126416 non-null  object 
 10  X-From                     495554 non-null  object 
 11  X-To                       495554 non-null  object 
 12  X-cc                       127172 non-null  object 
 13  X-bcc                      16

In [164]:
emails_df.isnull().sum()

Message-ID                    21847
Date                          21847
From                          21847
To                            21847
Subject                       38515
Cc                           393139
Mime-Version                  21847
Content-Type                  21847
Content-Transfer-Encoding     21847
Bcc                          390985
X-From                        21847
X-To                          21847
X-cc                         390229
X-bcc                        517233
X-Folder                      21847
X-Origin                      21847
X-FileName                    22394
Message-Body                  21848
dtype: int64

In [165]:
emails_df.dropna(how='all', inplace=True)
emails_df.dropna(subset=['Message-Body'], inplace=True)

In [166]:
# drop columns with lots of missing values
print(emails_df['Bcc'].isnull().mean() * 100, '% of X-bcc is empty')
print(emails_df['Cc'].isnull().mean() * 100, '% of X-cc is empty')
print(emails_df['X-bcc'].isnull().mean() * 100, '% of X-bcc is empty')
print(emails_df['X-cc'].isnull().mean() * 100, '% of X-cc is empty')

emails_df.drop(columns=['Bcc', 'Cc', 'X-bcc', 'X-cc'], inplace=True)
print('remaining cols: ', emails_df.columns)

emails_df.reset_index(drop=True, inplace=True)

# emails_df.to_csv('../data/02_emails_handled_missing.csv', index=False)

74.4899132887905 % of X-bcc is empty
74.92457920747125 % of X-cc is empty
99.96609847988005 % of X-bcc is empty
74.33735644825074 % of X-cc is empty
remaining cols:  Index(['Message-ID', 'Date', 'From', 'To', 'Subject', 'Mime-Version',
       'Content-Type', 'Content-Transfer-Encoding', 'X-From', 'X-To',
       'X-Folder', 'X-Origin', 'X-FileName', 'Message-Body'],
      dtype='object')


In [167]:
# Impute with No Subject
emails_df['Subject'].fillna('No Subject', inplace=True)

In [168]:
# Drop columns that are not useful for prediction
emails_df = emails_df.drop(columns=['Message-ID', 'Mime-Version', 'Content-Type', 'Content-Transfer-Encoding', 'X-Folder', 'X-Origin', 'X-FileName'])

In [175]:
emails_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 323856 entries, 0 to 323855
Data columns (total 8 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   Date               323856 non-null  object
 1   From               323856 non-null  object
 2   To                 323856 non-null  object
 3   Subject            323856 non-null  object
 4   X-From             323856 non-null  object
 5   X-To               323856 non-null  object
 6   Message-Body       323856 non-null  object
 7   has_reply_forward  323856 non-null  bool  
dtypes: bool(1), object(7)
memory usage: 17.6+ MB


In [169]:
print(emails_df['Message-Body'][::16000])

0                                 Here is our forecast\n\n 
16000     in\n\n\n   \n\n\nFrom:  Bryan Hull            ...
32000     The Commission issued an order on May 8, 2001 ...
48000     Hey Dirk,\n\n Could you tell me where I would ...
64000     713 646-8525\n\n -----Original Message-----\nF...
80000      The Ron Brown Scholarships are available for ...
96000     The volume for the 24th is in error.  It is re...
112000    ---------------------- Forwarded by Drew Fossu...
128000    Hey, I'm too tired to think so I started going...
144000    Deal 457758.1 for 11/14/00, HE 7 shows 25mw @ ...
160000    Myself,  being of Mexican descent there is no ...
176000    Susan/Tana,\n\nWe have recently traded a deal ...
192000    ---------------------- Forwarded by Vince J Ka...
208000    Our proposal was accepted. Dust off your San F...
224000    I'll send it back on Tuesday.\n\n\n\n\nJose Be...
240000    Attached please find two documents for Friday'...
256000    Wait until Louise gets back.  

In [170]:
print(emails_df['Message-Body'][192000])

---------------------- Forwarded by Vince J Kaminski/HOU/ECT on 09/06/2000 
03:38 PM ---------------------------


"Les Clewlow" <Les@lacima.co.uk> on 09/05/2000 09:30:02 PM
To: "chris" <chris@lacima.co.uk>, "Vince J Kaminski" 
<Vince_J_Kaminski@ei.enron.com>
cc:  
Subject: Fw: EPRM



----- Original Message -----
From: Dave Hall <dhall@riskwaters.com>
To: <les@lacima.co.uk>
Sent: Tuesday, September 05, 2000 8:53 PM
Subject: EPRM


> Dear Mr Clewlow
>
> Thanks for your piece on Var for inclusion in the October edition of
Energy
> & Power
> Risk Management.
>
> The article is attached. There are a few queries from the editor and
myself
> written in bold in the text. You might find that some parts of the piece
> have been edited to
> conform to our 'house style', etc.
>
> Comments and suggestions welcome.
>
> Should we send this article to anybody else for their approval?
>
> Thanks and kind regards,
>
>
> Dave Hall
> Chief Subeditor
> Energy & Power Risk Management magazine
> Risk Water

In [171]:
emails_df['Message-Body']

0                                 Here is our forecast\n\n 
1         Traveling to have a business meeting takes the...
2                            test successful.  way to go!!!
3         Randy,\n\n Can you send me a schedule of the s...
4                       Let's shoot for Tuesday at 11:45.  
                                ...                        
495548    This is a trade with OIL-SPEC-HEDGE-NG (John L...
495549    Some of my position is with the Alberta Term b...
495550    2\n\n -----Original Message-----\nFrom: \tDouc...
495551    Analyst\t\t\t\t\tRank\n\nStephane Brodeur\t\t\...
495552    i think the YMCA has a class that is for peopl...
Name: Message-Body, Length: 495553, dtype: object

In [172]:
# Find the emails that have common reply or forward patterns
# -+\s*(Original|Forwarded)
def find_reply_forward(msg_body):

    if not isinstance(msg_body, str):
        return False

    pattern = re.compile(
        r"""(?i)
        -+\s*(Original|Forwarded # forwarded
        |On\s.+wrote: # replies
        |>\s*From: # Quoted replies
        |>\s*Sent:
        |>\s*To:
        |Note:\s*forwarded\s*message\s*attached  # Forwarded note
        |Begin forwarded message
        |Message forwarded
        )""",
        re.IGNORECASE | re.VERBOSE | re.DOTALL)
    return bool(pattern.search(msg_body))

emails_df['has_reply_forward'] = emails_df['Message-Body'].apply(find_reply_forward)


In [173]:
emails_df['has_reply_forward'].value_counts()

False    323856
True     171697
Name: has_reply_forward, dtype: int64

In [174]:
# Remove the emails that have reply or forward
emails_df = emails_df[emails_df['has_reply_forward'] == False].reset_index(drop=True)
emails_df['has_reply_forward'].value_counts()



False    323856
Name: has_reply_forward, dtype: int64